# Latex Accuracy Delta Table

Configure `RUN_ROWS`, run the loading cell, then run the final cell to display and print the LaTeX table. Each task value is `final accuracy - original accuracy`, where the original accuracy is the first row in the task JSONL file and the final accuracy is the last row. The `Average` column excludes the task the run was trained on.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import re
import warnings

import pandas as pd
from IPython.display import display


REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "evaluation").is_dir() and (candidate / "tasks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the es_finetuning repo.")

SEARCH_ROOTS = [
    REPO_ROOT / "experiments" / "test",
    REPO_ROOT / "grpo_experiments" / "runs",
]
DISCOVERY_DEPTH = 2

TASKS = [
    "countdown",
    "gsm8k",
    "proofwriter",
    "hellaswag",
    "piqa",
    "arc-challenge",
    "mmlu-pro",
]
TASK_LABELS = {
    "countdown": "Countdown",
    "gsm8k": "GSM8K",
    "proofwriter": "ProofWriter",
    "hellaswag": "HellaSwag",
    "piqa": "PIQA",
    "arc-challenge": "ARC-C",
    "mmlu-pro": "MMLU-Pro",
}

# Add one entry per table row. The notebook will automatically find matching
# ES runs with the same name after removing seed number and final datetime.
# GRPO rows are discovered under grpo_experiments/runs and kept separate from ES rows.

# Countdown
# RUN_ROWS = [
#     {
#         "label": "Qwen2.5 (ES)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize30_msFalse_20260315_212803",
#     },
#     {
#         "label": "Qwen2.5 (GRPO)",
#         "run": "Qwen2.5-3B-Instruct_grpo_countdown_seed42",
#     },
#     {
#         "label": "Llama-3.2 (ES)",
#         "run": "Llama-3.2-3B-Instruct_countdown_seed0_popsize30_msFalse_20260428_154409",
#     },
#     {
#         "label": "Llama-3.2 (GRPO)",
#         "run": "Llama-3.2-3B-Instruct_grpo_countdown",
#     },
# ]

# Proofwriter
# RUN_ROWS = [
#     {
#         "label": "Qwen2.5 (ES)",
#         "run": "Qwen2.5-3B-Instruct_proofwriter_seed0_popsize30_msFalse_20260328_203835",
#     },
#     {
#         "label": "Qwen2.5 (GRPO)",
#         "run": "Qwen2.5-3B-Instruct_grpo_proofwriter",
#     },
#     {
#         "label": "Llama-3.2 (ES)",
#         "run": "Llama-3.2-3B-Instruct_proofwriter_seed0_popsize30_msFalse_20260501_055633",
#     },
#     {
#         "label": "Llama-3.2 (GRPO)",
#         "run": "Llama-3.2-3B-Instruct_grpo_proofwriter",
#     },
# ]

# RUN_ROWS = [
#     {
#         "label": "ES (30)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize30_msFalse_20260315_212803",
#     },
#     {
#         "label": "ES+AWD (L1)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize30_msFalse_l1wd0.01_20260320_201242",
#     },
#     {
#         "label": "ES+AWD (L2)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize30_msFalse_l2wd10.0_20260421_151321",
#     },
# ]

# RUN_ROWS = [
#     {
#         "label": "ES (30)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize30_msFalse_20260315_212803",
#     },
#     {
#         "label": "ES (128)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize128_msFalse_20260403_143926",
#     },
#     {
#         "label": "ES (256)",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize256_msFalse_20260427_033636",
#     },
# ]

# RUN_ROWS = [
#     {
#         "label": "ES 1.5B",
#         "run": "Qwen2.5-1.5B-Instruct_countdown_seed0_popsize30_msFalse_20260316_120019",
#     },
#     {
#         "label": "ES 3B",
#         "run": "Qwen2.5-3B-Instruct_countdown_seed0_popsize30_msFalse_20260315_212803",
#     },
#     {
#         "label": "ES 7B",
#         "run": "Qwen2.5-7B-Instruct_countdown_seed0_popsize30_msFalse_20260325_085603",
#     },
#     {
#         "label": "GRPO 1.5B",
#         "run": "Qwen2.5-1.5B-Instruct_grpo_countdown_seed42",
#     },
#     {
#         "label": "GRPO 3B",
#         "run": "Qwen2.5-3B-Instruct_grpo_countdown_seed42",
#     },
#     {
#         "label": "GRPO 7B",
#         "run": "Qwen2.5-7B-Instruct_grpo_countdown",
#     },
# ]

# RUN_ROWS = [
#     {
#         "label": "ES 1.5B",
#         "run": "Qwen2.5-1.5B-Instruct_proofwriter_seed0_popsize30_msFalse_20260504_064942",
#     },
#     {
#         "label": "ES 3B",
#         "run": "Qwen2.5-3B-Instruct_proofwriter_seed0_popsize30_msFalse_20260328_203835",
#     },
#     {
#         "label": "ES 7B",
#         "run": "Qwen2.5-7B-Instruct_proofwriter_seed0_popsize30_msFalse_20260401_135528",
#     },
#     {
#         "label": "GRPO 1.5B",
#         "run": "Qwen2.5-1.5B-Instruct_grpo_proofwriter",
#     },
#     {
#         "label": "GRPO 3B",
#         "run": "Qwen2.5-3B-Instruct_grpo_proofwriter",
#     },
#     {
#         "label": "GRPO 7B",
#         "run": "Qwen2.5-7B-Instruct_grpo_proofwriter",
#     },
# ]

# If the same seed was run more than once with different final datetimes,
# keep the latest datetime so a rerun does not double-count one seed.
DUPLICATE_SEED_POLICY = "latest"
AVERAGE_REQUIRES_ALL_TASKS = False

In [2]:
SEED_TOKEN_RE = re.compile(r"seed(\d+)$")
DATE_TOKEN_RE = re.compile(r"\d{8}$")
TIME_TOKEN_RE = re.compile(r"\d{6}$")


def detect_backend(run_dir: Path) -> str | None:
    if (run_dir / "args.json").is_file() and (run_dir / "iteration_updates").is_dir():
        return "es"
    if (run_dir / "model.txt").is_file():
        for child in run_dir.iterdir():
            if child.is_dir() and parse_global_step(child.name) is not None:
                return "grpo"
    return None


def parse_global_step(name: str) -> int | None:
    match = re.search(r"_global_step_(\d+)$", name)
    if match is None:
        return None
    return int(match.group(1))


def parse_seed(name: str) -> int | None:
    for token in name.split("_"):
        match = SEED_TOKEN_RE.fullmatch(token)
        if match is not None:
            return int(match.group(1))
    return None


def parse_datetime_suffix(name: str) -> tuple[str, str] | None:
    tokens = name.split("_")
    if len(tokens) >= 2 and DATE_TOKEN_RE.fullmatch(tokens[-2]) and TIME_TOKEN_RE.fullmatch(tokens[-1]):
        return tokens[-2], tokens[-1]
    return None


def canonical_family_name(name: str) -> str:
    tokens = name.split("_")
    if len(tokens) >= 2 and DATE_TOKEN_RE.fullmatch(tokens[-2]) and TIME_TOKEN_RE.fullmatch(tokens[-1]):
        tokens = tokens[:-2]
    tokens = [token for token in tokens if SEED_TOKEN_RE.fullmatch(token) is None]
    return "_".join(tokens)


def infer_train_task(identifier: str) -> str:
    normalized_identifier = identifier.replace("_", "-").lower()
    matches = [task for task in TASKS if task.replace("_", "-") in normalized_identifier]
    if not matches:
        raise ValueError(f"Could not infer train task from run identifier: {identifier}")
    if len(matches) > 1:
        raise ValueError(f"Ambiguous train task for run identifier {identifier}: {matches}")
    return matches[0]


def seed_label(run_dir: Path) -> str:
    seed = parse_seed(run_dir.name)
    if seed is None:
        return run_dir.name
    return f"seed{seed}"


def iter_candidate_run_dirs(root: Path, max_depth: int):
    if not root.exists():
        return

    def visit(path: Path, depth: int):
        backend = detect_backend(path)
        if backend is not None:
            yield path.resolve(), backend
            return
        if depth <= 0:
            return
        for child in sorted(path.iterdir()):
            if child.is_dir():
                yield from visit(child, depth - 1)

    yield from visit(root, max_depth)


def discover_runs(search_roots: list[Path], max_depth: int = DISCOVERY_DEPTH) -> dict[Path, str]:
    run_index: dict[Path, str] = {}
    for root in search_roots:
        for run_dir, backend in iter_candidate_run_dirs(root, max_depth):
            run_index[run_dir] = backend
    return dict(sorted(run_index.items(), key=lambda item: str(item[0])))


def normalize_row_spec(row_spec) -> dict[str, str]:
    if isinstance(row_spec, (str, Path)):
        return {"label": Path(row_spec).name, "run": str(row_spec)}
    run = row_spec.get("run") or row_spec.get("path") or row_spec.get("name")
    if run is None:
        raise ValueError(f"Row spec is missing a run/path/name field: {row_spec}")
    label = row_spec.get("label") or Path(str(run)).name
    return {"label": str(label), "run": str(run)}


def resolve_configured_run(row_spec, run_index: dict[Path, str]) -> Path:
    spec = normalize_row_spec(row_spec)
    run_text = spec["run"]
    path = Path(run_text).expanduser()
    candidates = []
    if path.is_absolute():
        candidates.append(path.resolve())
    else:
        candidates.append((REPO_ROOT / path).resolve())

    for candidate in candidates:
        if candidate in run_index:
            return candidate

    exact_name_matches = [run_dir for run_dir in run_index if run_dir.name == run_text]
    if len(exact_name_matches) == 1:
        return exact_name_matches[0]
    if len(exact_name_matches) > 1:
        match_list = "\n".join(str(path) for path in exact_name_matches)
        raise ValueError(f"Run name {run_text!r} is ambiguous. Matches:\n{match_list}")

    raise FileNotFoundError(f"Could not resolve configured run: {run_text}")


def select_one_run_per_seed(run_dirs: list[Path]) -> tuple[list[Path], dict[str, list[Path]]]:
    grouped: dict[str, list[Path]] = defaultdict(list)
    for run_dir in run_dirs:
        grouped[seed_label(run_dir)].append(run_dir)

    selected = []
    duplicates = {}
    for key, paths in sorted(grouped.items()):
        if len(paths) == 1 or DUPLICATE_SEED_POLICY == "all":
            selected.extend(sorted(paths, key=lambda path: path.name))
            continue

        duplicates[key] = sorted(paths, key=lambda path: path.name)
        if DUPLICATE_SEED_POLICY != "latest":
            raise ValueError(f"Unsupported DUPLICATE_SEED_POLICY: {DUPLICATE_SEED_POLICY}")
        selected.append(max(paths, key=lambda path: (parse_datetime_suffix(path.name) or ("", ""), path.name)))

    return sorted(selected, key=lambda path: path.name), duplicates


def matching_family_runs(anchor_run_dir: Path, run_index: dict[Path, str]) -> tuple[list[Path], dict[str, list[Path]]]:
    backend = run_index[anchor_run_dir]
    family = canonical_family_name(anchor_run_dir.name)
    matches = [
        run_dir
        for run_dir, run_backend in run_index.items()
        if run_backend == backend and canonical_family_name(run_dir.name) == family
    ]
    return select_one_run_per_seed(matches)


def load_jsonl_records(path: Path) -> list[dict]:
    records = []
    for line in path.read_text().splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def empty_metric(path: Path, status: str) -> dict:
    return {
        "path": path,
        "original_accuracy": pd.NA,
        "final_accuracy": pd.NA,
        "accuracy_delta": pd.NA,
        "final_step": pd.NA,
        "max_step": pd.NA,
        "status": status,
    }


def load_accuracy_delta(run_dir: Path, task: str) -> dict:
    path = run_dir / f"{task}.jsonl"
    if not path.is_file():
        return empty_metric(path, "missing")

    records = load_jsonl_records(path)
    if not records:
        return empty_metric(path, "empty")

    frame = pd.DataFrame(records)
    if "accuracy" not in frame.columns:
        return empty_metric(path, "missing_accuracy")

    frame = frame.copy()
    frame["_row_order"] = range(len(frame))
    if "step" not in frame.columns:
        frame["step"] = frame["_row_order"]
    frame["step"] = pd.to_numeric(frame["step"], errors="coerce")
    first = frame.iloc[0]
    last = frame.iloc[-1]
    original_accuracy = float(first["accuracy"])
    final_accuracy = float(last["accuracy"])
    max_step = frame["step"].max()
    return {
        "path": path,
        "original_accuracy": original_accuracy,
        "final_accuracy": final_accuracy,
        "accuracy_delta": final_accuracy - original_accuracy,
        "final_step": int(last["step"]) if pd.notna(last["step"]) else pd.NA,
        "max_step": int(max_step) if pd.notna(max_step) else pd.NA,
        "status": "loaded",
    }


def mean_or_na(values: list[float]) -> float | pd._libs.missing.NAType:
    clean_values = [float(value) for value in values if pd.notna(value)]
    if not clean_values:
        return pd.NA
    return float(pd.Series(clean_values).mean())


def std_or_na(values: list[float]) -> float | pd._libs.missing.NAType:
    clean_values = [float(value) for value in values if pd.notna(value)]
    if len(clean_values) < 2:
        return pd.NA
    return float(pd.Series(clean_values).std(ddof=1))

In [ ]:
run_index = discover_runs(SEARCH_ROOTS)
print(f"Discovered {len(run_index)} run directories.")

table_rows = []
table_std_rows = []
table_count_rows = []
summary_rows = []
raw_metric_rows = []

for row_spec in RUN_ROWS:
    spec = normalize_row_spec(row_spec)
    anchor_run_dir = resolve_configured_run(spec, run_index)
    backend = run_index[anchor_run_dir]
    family = canonical_family_name(anchor_run_dir.name)
    train_task = infer_train_task(family)
    matched_run_dirs, duplicates = matching_family_runs(anchor_run_dir, run_index)
    n_seeds = len(matched_run_dirs)

    if duplicates:
        for duplicate_seed, duplicate_paths in duplicates.items():
            kept = [path for path in matched_run_dirs if seed_label(path) == duplicate_seed]
            warnings.warn(
                f"{spec['label']}: found {len(duplicate_paths)} runs for {duplicate_seed}; "
                f"keeping {kept[0].name if kept else 'none'}"
            )

    row = {"Run": spec["label"]}
    std_row = {"Run": spec["label"]}
    count_row = {"Run": spec["label"]}
    seed_transfer_values: dict[str, list[float]] = defaultdict(list)
    max_steps_for_print = []

    for task in TASKS:
        loaded_deltas = []
        loaded_steps = []
        missing_seed_labels = []

        for run_dir in matched_run_dirs:
            current_seed_label = seed_label(run_dir)
            metric = load_accuracy_delta(run_dir, task)
            raw_metric_rows.append(
                {
                    "run": spec["label"],
                    "backend": backend,
                    "family": family,
                    "train_task": train_task,
                    "seed": current_seed_label,
                    "run_dir": str(run_dir.relative_to(REPO_ROOT)),
                    "task": task,
                    "original_accuracy": metric["original_accuracy"],
                    "final_accuracy": metric["final_accuracy"],
                    "accuracy_delta": metric["accuracy_delta"],
                    "final_step": metric["final_step"],
                    "max_step": metric["max_step"],
                    "status": metric["status"],
                    "metrics_path": str(metric["path"].relative_to(REPO_ROOT)),
                }
            )
            if metric["status"] == "loaded":
                loaded_deltas.append(metric["accuracy_delta"])
                loaded_steps.append(metric["max_step"])
                if task != train_task:
                    seed_transfer_values[current_seed_label].append(metric["accuracy_delta"])
            else:
                missing_seed_labels.append(current_seed_label)

        task_mean = mean_or_na(loaded_deltas)
        row[TASK_LABELS[task]] = task_mean
        std_row[TASK_LABELS[task]] = std_or_na(loaded_deltas)
        count_row[TASK_LABELS[task]] = len(loaded_deltas)

        max_step = max(loaded_steps) if loaded_steps else pd.NA
        max_steps_for_print.append(f"{TASK_LABELS[task]}={max_step if pd.notna(max_step) else 'NA'}")
        summary_rows.append(
            {
                "run": spec["label"],
                "backend": backend,
                "family": family,
                "train_task": TASK_LABELS[train_task],
                "task": TASK_LABELS[task],
                "n_seeds_found": n_seeds,
                "seeds_with_metric": len(loaded_deltas),
                "max_step": max_step,
                "missing_or_empty": ", ".join(missing_seed_labels),
            }
        )

    seed_average_values = []
    required_transfer_task_count = len(TASKS) - 1
    for transfer_values in seed_transfer_values.values():
        if AVERAGE_REQUIRES_ALL_TASKS and len(transfer_values) != required_transfer_task_count:
            continue
        seed_average = mean_or_na(transfer_values)
        if pd.notna(seed_average):
            seed_average_values.append(seed_average)

    if AVERAGE_REQUIRES_ALL_TASKS and len(seed_average_values) != n_seeds:
        row["Average"] = pd.NA
        std_row["Average"] = pd.NA
        count_row["Average"] = len(seed_average_values)
    else:
        row["Average"] = mean_or_na(seed_average_values)
        std_row["Average"] = std_or_na(seed_average_values)
        count_row["Average"] = len(seed_average_values)
    table_rows.append(row)
    table_std_rows.append(std_row)
    table_count_rows.append(count_row)

    print(f"{spec['label']}: {n_seeds} seed(s) found; average excludes {TASK_LABELS[train_task]}; highest loaded steps: {', '.join(max_steps_for_print)}")

table_df = pd.DataFrame(table_rows)
table_std_df = pd.DataFrame(table_std_rows)
table_count_df = pd.DataFrame(table_count_rows)
load_summary_df = pd.DataFrame(summary_rows)
raw_metrics_df = pd.DataFrame(raw_metric_rows)

display(load_summary_df)
display(raw_metrics_df)
display(table_df)

In [ ]:
LATEX_ESCAPE_REPLACEMENTS = {
    "\\": r"\textbackslash{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "_": r"\_",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
    "^": r"\textasciicircum{}",
}


def latex_escape_text(value) -> str:
    return "".join(LATEX_ESCAPE_REPLACEMENTS.get(char, char) for char in str(value))


def format_delta(value, std_value=pd.NA, count: int = 1, digits: int = 1) -> str:
    if pd.isna(value):
        return "--"
    mean_text = f"{float(value):+.{digits}f}"
    if count > 1 and pd.notna(std_value):
        std_text = f"{float(std_value):.{digits}f}"
        return f"${mean_text}_{{({std_text})}}$"
    return f"${mean_text}$"


def make_latex_table(
    table: pd.DataFrame,
    std_table: pd.DataFrame | None = None,
    count_table: pd.DataFrame | None = None,
    digits: int = 1,
    caption: str | None = None,
    label: str | None = None,
) -> str:
    latex_df = table.copy()
    if "Seeds" in latex_df.columns:
        latex_df = latex_df.drop(columns=["Seeds"])
    latex_df["Run"] = latex_df["Run"].map(latex_escape_text)
    accuracy_columns = [TASK_LABELS[task] for task in TASKS] + ["Average"]
    for column in accuracy_columns:
        latex_df[column] = [
            format_delta(
                mean_value,
                std_value=std_table.at[index, column] if std_table is not None else pd.NA,
                count=int(count_table.at[index, column]) if count_table is not None and pd.notna(count_table.at[index, column]) else 1,
                digits=digits,
            )
            for index, mean_value in table[column].items()
        ]
    return latex_df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
    )


latex_table = make_latex_table(table_df, table_std_df, table_count_df, digits=1)
print(latex_table)